# Notebook 08 — Molecular Datasets: Loading, Processing, and Managing Chemical Libraries

Real-world cheminformatics means working with **datasets** — from a handful of screening hits you pulled off an HPLC run, to large vendor catalogs with millions of compounds. The gap between "I can parse one molecule" and "I can wrangle a whole library" is where most projects stall.

This notebook covers the practical plumbing:

1. **Reading & writing** molecular data in common formats (SMILES, CSV, SDF)
2. **Integrating** RDKit with pandas for tabular workflows
3. **Curating** messy real-world data — salt stripping, charge neutralization, deduplication

Think of this as the cheminformatics equivalent of setting up your lab notebook, labeling your vials, and making sure your reagents are pure before you start a synthesis. All data is **embedded** — no external downloads required.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Draw
from rdkit.Chem import PandasTools
import pandas as pd
from io import StringIO
import tempfile
import os

## 1. An Embedded Drug Dataset

Instead of downloading from ChEMBL or PubChem (which adds fragile network dependencies), we embed ~50 well-known drugs directly as SMILES strings. You have almost certainly handled many of these compounds in a wet-lab setting — aspirin, ibuprofen, metformin, statins, beta-blockers, antibiotics.

**Why SMILES?** SMILES (Simplified Molecular-Input Line-Entry System) is the most compact text representation of a molecule. One line of ASCII encodes a full molecular graph. It is the CSV of chemistry — not pretty, but universal and easy to move around.

In [ ]:
KNOWN_DRUGS = [
    ("Aspirin", "CC(=O)OC1=CC=CC=C1C(=O)O"),
    ("Ibuprofen", "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"),
    ("Acetaminophen", "CC(=O)NC1=CC=C(O)C=C1"),
    ("Caffeine", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"),
    ("Metformin", "CN(C)C(=N)NC(=N)N"),
    ("Omeprazole", "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1"),
    ("Atorvastatin", "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1"),
    ("Metoprolol", "COCCOCC(O)CNC(C)C"),
    ("Amlodipine", "CCOC(=O)C1=C(COCCN)NC(C)=C(C1C1=CC=CC=C1Cl)C(=O)OC"),
    ("Lisinopril", "NCCCC[C@@H](N[C@@H](CCC1=CC=CC=C1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O"),
    ("Simvastatin", "CCC(C)(C)C(=O)O[C@H]1C[C@@H](O)C=C2C=C[C@H](C)[C@H](CC[C@@H](O)CC(=O)O)[C@@H]21"),
    ("Losartan", "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO"),
    ("Warfarin", "CC(=O)CC(C1=CC=CC=C1)C1=C(O)C2=CC=CC=C2OC1=O"),
    ("Ciprofloxacin", "OC(=O)C1=CN(C2CC2)C2=CC(N3CCNCC3)=C(F)C=C2C1=O"),
    ("Amoxicillin", "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)C3=CC=C(O)C=C3)C(=O)N2[C@@H]1C(=O)O"),
    ("Diazepam", "CN1C(=O)CN=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C21"),
    ("Fluoxetine", "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1"),
    ("Sertraline", "CN[C@H]1CC[C@@H](C2=CC(Cl)=C(Cl)C=C2)C2=CC=CC=C12"),
    ("Cetirizine", "OC(=O)COCCN1CCN(CC1)C(C1=CC=CC=C1)C1=CC=C(Cl)C=C1"),
    ("Loratadine", "CCOC(=O)N1CCC(=C2C3=CC=C(Cl)C=C3CCC3=CC=CN=C32)CC1"),
    ("Ranitidine", "CNC(/N=C/[N+](=O)[O-])NCCSCC1=CC=C(CN(C)C)O1"),
    ("Sildenafil", "CCCC1=NN(C)C2=C1NC(=NC2=O)C1=CC(=CC=C1OCC)S(=O)(=O)N1CCN(C)CC1"),
    ("Tamoxifen", "CCC(=C(C1=CC=CC=C1)C1=CC=C(OCCN(C)C)C=C1)C1=CC=CC=C1"),
    ("Dexamethasone", "C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C@@]3(F)[C@@H](O)C[C@]2(C)[C@@]1(O)C(=O)CO"),
    ("Prednisone", "C[C@]12C=CC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[C@]2(C)[C@H]1CC[C@@]2(O)C(=O)CO"),
    ("Methotrexate", "CN(CC1=CN=C2N=C(N)N=C(N)C2=N1)C1=CC=C(C=C1)C(=O)N[C@@H](CCC(=O)O)C(=O)O"),
    ("Chloroquine", "CCN(CC)CCCC(C)NC1=CC=NC2=CC(Cl)=CC=C12"),
    ("Penicillin V", "CC1(C)S[C@@H]2[C@H](NC(=O)COC3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O"),
    ("Naproxen", "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"),
    ("Diclofenac", "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl"),
    ("Clopidogrel", "COC(=O)[C@H](C1=CC=CS1)C1=CC2=CC=CC=C2N1"),
    ("Montelukast", "CC(C)(O)C1=CC=CC=C1CCC(SCC1(CC1)CC(=O)O)C1=CC=CC(=C1)/C=C/C1=CC2=CC=CC=C2C=C1"),
    ("Valsartan", "CCCCC(=O)N(CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)[C@@H](C(C)C)C(=O)O"),
    ("Esomeprazole", "COC1=CC2=C(C=C1C)NC(=N2)[S@@](=O)CC1=CC=C(OC)C(C)=N1"),
    ("Pantoprazole", "COC1=CC=NC(CS(=O)C2=NC3=C(N2)C=C(OC(F)F)C=C3)=C1OC"),
    ("Citalopram", "N#CCC1(C2=CC=C(F)C=C2)OCC2=CC(=CC=C21)CN1CCCC1"),
    ("Gabapentin", "NCC1(CC(=O)O)CCCCC1"),
    ("Hydrochlorothiazide", "NS(=O)(=O)C1=CC2=C(NCNS2(=O)=O)C=C1Cl"),
    ("Furosemide", "NS(=O)(=O)C1=CC(C(=O)O)=C(NCC2=CC=CO2)C=C1Cl"),
    ("Allopurinol", "O=C1N=CN=C2NNC=C12"),
    ("Levothyroxine", "N[C@@H](CC1=CC(I)=C(OC2=CC(I)=C(O)C(I)=C2)C(I)=C1)C(=O)O"),
    ("Azithromycin", "CC[C@H]1OC(=O)[C@H](C)[C@@H](O[C@H]2C[C@@](C)(OC)[C@@H](O)[C@H](C)O2)[C@H](C)[C@@H](O[C@@H]2O[C@H](C)C[C@H](N(C)C)[C@@H]2O)[C@](C)(O)C[C@@H](C)CN(C)[C@H](C)[C@@H](O)[C@]1(C)O"),
    ("Clarithromycin", "CC[C@H]1OC(=O)[C@H](C)[C@@H](O[C@H]2C[C@@](C)(OC)[C@@H](O)[C@H](C)O2)[C@H](C)[C@@H](O[C@@H]2O[C@H](C)C[C@H](N(C)C)[C@@H]2O)[C@](C)(OC)C[C@@H](C)C(=O)[C@H](C)[C@@H](O)[C@]1(C)O"),
    ("Atenolol", "CC(C)NCC(O)COC1=CC=C(CC(N)=O)C=C1"),
    ("Propranolol", "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12"),
    ("Carvedilol", "COC1=CC=CC=C1OCCNCC(O)COC1=CC=CC2=C1C=CC(=O)N2"),
    ("Duloxetine", "CNCC(OC1=CC=CC2=CC=CC=C12)C1=CC=CS1"),
    ("Venlafaxine", "COC1=CC=C(C(CN(C)C)C2(O)CCCCC2)C=C1"),
    ("Rosuvastatin", "CC(C1=NC(N(C)S(=O)(=O)C)=NC(=C1/C=C/[C@@H](O)C[C@@H](O)CC(=O)O)C1=CC=C(F)C=C1)C"),
]

print(f"Embedded dataset: {len(KNOWN_DRUGS)} known drugs")

## 2. Reading SMILES and CSV Data

### Chemistry refresher: file formats as containers

In the wet lab, you keep compound identifiers in spreadsheets, databases, or even just labels on vials. Computationally, the two simplest containers are:

- **`.smi` files** — one SMILES per line, optionally followed by a name (tab- or space-separated). RDKit reads these with `SmilesMolSupplier`.
- **CSV / TSV files** — tabular data with a SMILES column. Best handled through **pandas** + `Chem.MolFromSmiles`, which gives you full DataFrame power (filtering, grouping, joins).

The pattern is always the same: text in, molecule objects out. If parsing fails (bad SMILES, kekulization errors), RDKit returns `None` rather than crashing — you need to check for this.

In [ ]:
# Build a DataFrame from our embedded data
df = pd.DataFrame(KNOWN_DRUGS, columns=["Name", "SMILES"])
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)

# Check for parsing failures — this is your first quality gate
n_failed = df["mol"].isna().sum()
print(f"Successfully parsed: {len(df) - n_failed}/{len(df)}")
df = df.dropna(subset=["mol"])
print(f"Working with {len(df)} molecules")
df.head()

### Reading from a `.smi` file with `SmilesMolSupplier`

When you receive a SMILES file from a collaborator or vendor, `SmilesMolSupplier` reads it directly without needing pandas. We write a temp file to demonstrate the round-trip.

In [ ]:
# Write a .smi file from the first 10 drugs, then read it back
smi_content = "\n".join(f"{smi} {name}" for name, smi in KNOWN_DRUGS[:10])

with tempfile.NamedTemporaryFile(mode='w', suffix='.smi', delete=False) as f:
    f.write(smi_content)
    smi_path = f.name

print(f"Wrote {smi_path}\n")

# SmilesMolSupplier reads the file lazily (one molecule at a time)
supplier = Chem.SmilesMolSupplier(smi_path, titleLine=False)
for mol in supplier:
    if mol:
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else "unnamed"
        print(f"  {name:20s}  {Chem.MolToSmiles(mol)}")

os.unlink(smi_path)
print(f"\nCleaned up temp file.")

## 3. Reading and Writing SDF Files

### Chemistry refresher: SDF as the universal exchange format

If SMILES is the CSV of chemistry, then **SDF (Structure-Data File)** is the Excel workbook. Each record in an SDF contains:

1. A **connection table (molblock)** — atom coordinates, bond orders, stereochemistry. This is richer than SMILES because it includes 2D or 3D coordinates.
2. **Data fields (tags)** — arbitrary key-value pairs like `MW`, `IC50`, `vendor_id`.

You encounter SDFs when:
- Downloading compound libraries from **ChEMBL**, **PubChem**, or **ZINC**
- Receiving results from a **CRO** (contract research organization)
- Exchanging data with **docking software** (Glide, AutoDock, etc.)

RDKit provides `SDWriter` to create SDFs and `SDMolSupplier` / `ForwardSDMolSupplier` to read them.

In [ ]:
# --- Write an SDF with computed properties ---
with tempfile.NamedTemporaryFile(suffix='.sdf', delete=False) as f:
    sdf_path = f.name

writer = Chem.SDWriter(sdf_path)
for _, row in df.head(10).iterrows():
    mol = row["mol"]
    AllChem.Compute2DCoords(mol)  # generate 2D layout for the molblock
    mol.SetProp("_Name", row["Name"])
    mol.SetProp("MW", f"{Descriptors.MolWt(mol):.2f}")
    mol.SetProp("LogP", f"{Descriptors.MolLogP(mol):.2f}")
    mol.SetProp("HBA", str(rdMolDescriptors.CalcNumHBA(mol)))
    mol.SetProp("HBD", str(rdMolDescriptors.CalcNumHBD(mol)))
    writer.write(mol)
writer.close()
print(f"Wrote SDF to {sdf_path}")

In [ ]:
# --- Read it back with SDMolSupplier ---
# SDMolSupplier loads the entire file into memory (random access by index)
supplier = Chem.SDMolSupplier(sdf_path)
print(f"Read {len(supplier)} molecules from SDF\n")

for mol in supplier:
    if mol:
        print(f"  {mol.GetProp('_Name'):20s}  MW={mol.GetProp('MW'):>8s}  LogP={mol.GetProp('LogP'):>6s}  HBA={mol.GetProp('HBA')}  HBD={mol.GetProp('HBD')}")

### `ForwardSDMolSupplier` — streaming large files

When your SDF is too large to fit in memory (think: full ChEMBL at ~2 million compounds), use `ForwardSDMolSupplier`. It reads molecules one at a time from an open file handle — forward-only, no random access, but constant memory usage. This is the difference between loading a whole Excel file vs. reading it row by row.

In [ ]:
# ForwardSDMolSupplier — streaming read (constant memory)
count = 0
with open(sdf_path, 'rb') as fh:
    fwd_supplier = Chem.ForwardSDMolSupplier(fh)
    for mol in fwd_supplier:
        if mol:
            count += 1

print(f"Streamed {count} molecules with ForwardSDMolSupplier")
print("(No len() available — forward-only iteration)")

# Clean up
os.unlink(sdf_path)

## 4. PandasTools — RDKit + pandas Integration

`PandasTools` bridges the gap between RDKit molecule objects and pandas DataFrames. The key function `AddMoleculeColumnToFrame` parses a SMILES column and adds an `ROMol` column that can render structures directly in Jupyter notebooks.

> **Note:** Molecule rendering in DataFrame output depends on your Jupyter environment and RDKit build. If you see `<rdkit.Chem.rdchem.Mol object>` instead of images, that is normal — the chemistry is still there, just not rendered visually. Use `Draw.MolToImage()` or `Draw.MolsToGridImage()` for explicit visualization.

In [ ]:
# Add an ROMol column via PandasTools (parses SMILES → mol objects)
PandasTools.AddMoleculeColumnToFrame(df, smilesCol="SMILES", molCol="ROMol")

# Compute common physicochemical descriptors
df["MW"] = df["mol"].apply(Descriptors.MolWt)
df["LogP"] = df["mol"].apply(Descriptors.MolLogP)
df["HBA"] = df["mol"].apply(rdMolDescriptors.CalcNumHBA)
df["HBD"] = df["mol"].apply(rdMolDescriptors.CalcNumHBD)
df["TPSA"] = df["mol"].apply(Descriptors.TPSA)
df["RotBonds"] = df["mol"].apply(rdMolDescriptors.CalcNumRotatableBonds)

# Display a clean summary
df[["Name", "MW", "LogP", "HBA", "HBD", "TPSA", "RotBonds"]].head(10)

In [ ]:
# Quick statistical summary of our drug library
print("Descriptor statistics for our 50-drug library:\n")
df[["MW", "LogP", "HBA", "HBD", "TPSA", "RotBonds"]].describe().round(2)

## 5. Data Curation — Cleaning Molecular Datasets

### Chemistry refresher: dirty data is like impure reagents

In the wet lab, you would never run a reaction with contaminated starting materials — you purify first. The same principle applies to computational chemistry:

| Wet-lab problem | Computational analog |
|---|---|
| Impure reagent | Invalid SMILES, failed parsing |
| Salt form vs. free base | Counterions included in the SMILES (e.g., `[Na+].[O-]C(=O)...`) |
| Protonation state depends on pH | Charged species that should be neutral at physiological pH |
| Duplicate vials in the freezer | Same molecule entered twice with different SMILES representations |

A curation pipeline typically runs these steps in order:
1. **Sanitization** — reject invalid structures
2. **Salt stripping** — keep only the largest (active) fragment
3. **Charge neutralization** — convert to the neutral form
4. **Canonicalization & deduplication** — one canonical SMILES per unique structure

### 5a. Sanitization — checking for valid chemistry

`Chem.MolFromSmiles` already runs basic sanitization (kekulization, valence checks, aromaticity perception). If it returns `None`, the input was unparseable. We wrap this in a helper function for clarity.

In [ ]:
def sanitize_mol(smi):
    """Attempt to sanitize a SMILES string, return None if invalid."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

# Test with a mix of valid and problematic inputs
test_smiles = [
    ("Ethanol", "CCO"),
    ("Garbage string", "invalid"),
    ("Neopentane", "C(C)(C)(C)(C)C"),      # pentavalent carbon — bad!
    ("Copper sulfate", "[Cu+2].[O-]S(=O)(=O)[O-]"),  # valid salt
    ("Empty", ""),
    ("Benzene", "c1ccccc1"),
]

print("Sanitization results:\n")
for name, smi in test_smiles:
    mol = sanitize_mol(smi)
    status = "Valid" if mol else "Invalid / unparseable"
    print(f"  {name:20s}  {smi:40s}  ->  {status}")

### 5b. Salt stripping — keep the largest fragment

Many drugs are sold as salts for stability or solubility (e.g., metformin **hydrochloride**, sodium **diclofenac**). In SMILES, salts appear as disconnected fragments separated by `.` — for example, `CC(=O)[O-].[Na+]` is sodium acetate.

For most cheminformatics tasks (fingerprints, descriptors, docking), you want the **active pharmaceutical ingredient (API)** only — the largest organic fragment. RDKit's `SaltRemover` handles this.

In [ ]:
from rdkit.Chem.SaltRemover import SaltRemover

remover = SaltRemover()

salt_examples = [
    ("Sodium acetate",       "CC(=O)[O-].[Na+]"),
    ("Aminopropane HCl",     "[NH3+]CCC.[Cl-]"),
    ("Aspirin (no salt)",    "CC(=O)OC1=CC=CC=C1C(=O)O"),
    ("Lidocaine HCl",        "CCN(CC)CC(=O)NC1=C(C)C=CC=C1C.[Cl-]"),
    ("Calcium carbonate",    "[Ca+2].[O-]C([O-])=O"),
]

print("Salt stripping results:\n")
for name, smi in salt_examples:
    mol = Chem.MolFromSmiles(smi)
    stripped = remover.StripMol(mol)
    print(f"  {name:25s}  {smi:45s}  ->  {Chem.MolToSmiles(stripped)}")

### 5c. Charge neutralization

After salt stripping, you may still have charged species — a carboxylate `[O-]` that was deprotonated, or a protonated amine `[NH3+]`. For consistent descriptor computation and fingerprinting, it is often useful to **neutralize** these to their uncharged forms.

This is analogous to working at a reference pH: in the wet lab, you might report pKa values and note which protonation state dominates at pH 7.4. Computationally, the `Uncharger` from `rdMolStandardize` does a simple neutralization pass.

In [ ]:
from rdkit.Chem.MolStandardize import rdMolStandardize

uncharger = rdMolStandardize.Uncharger()

charged_examples = [
    ("Acetate anion",      "CC([O-])=O"),
    ("Protonated amine",   "C[NH3+]"),
    ("Deprotonated acid",  "CC(=O)[O-]"),
    ("Zwitterionic amino acid", "[NH3+][C@@H](CC1=CC=CC=C1)C([O-])=O"),
    ("Neutral ethanol",    "CCO"),
]

print("Charge neutralization results:\n")
for name, smi in charged_examples:
    mol = Chem.MolFromSmiles(smi)
    neutral = uncharger.uncharge(mol)
    original_smi = Chem.MolToSmiles(mol)
    neutral_smi = Chem.MolToSmiles(neutral)
    changed = " (changed)" if original_smi != neutral_smi else " (unchanged)"
    print(f"  {name:30s}  {original_smi:30s}  ->  {neutral_smi}{changed}")

### 5d. Deduplication by canonical SMILES

Different SMILES strings can encode the same molecule. For example, `C(C)O` and `CCO` and `OCC` are all ethanol. RDKit's canonical SMILES provides a **unique** string for each molecule, which makes deduplication straightforward — just drop duplicate canonical SMILES.

This is like checking your compound inventory for duplicate entries: same molecule, different labels.

In [ ]:
# First, demonstrate that different SMILES can encode the same molecule
ethanol_variants = ["CCO", "C(C)O", "OCC", "[CH3][CH2][OH]"]
canonical = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in ethanol_variants]
print("All of these are ethanol:")
for orig, canon in zip(ethanol_variants, canonical):
    print(f"  {orig:20s}  ->  canonical: {canon}")
print(f"\nUnique canonical forms: {len(set(canonical))}\n")

# Now deduplicate our drug DataFrame
df["canonical"] = df["mol"].apply(Chem.MolToSmiles)
n_before = len(df)
df_dedup = df.drop_duplicates(subset="canonical")
n_after = len(df_dedup)
print(f"Before deduplication: {n_before}")
print(f"After deduplication:  {n_after}")
print(f"Removed:              {n_before - n_after} duplicate(s)")

### 5e. Putting it all together — a curation pipeline

In practice, you chain these steps into a single function that takes raw SMILES and returns clean, standardized molecules (or `None` for rejects). Here is a minimal but production-ready pipeline.

In [ ]:
def curate_smiles(smi):
    """
    Full curation pipeline: parse -> strip salts -> neutralize -> canonicalize.
    Returns (canonical_smiles, mol) or (None, None) if the input is invalid.
    """
    # Step 1: Parse and sanitize
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, None

    # Step 2: Strip salts (keep largest fragment)
    remover = SaltRemover()
    mol = remover.StripMol(mol)

    # Step 3: Neutralize charges
    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)

    # Step 4: Canonicalize
    canonical = Chem.MolToSmiles(mol)

    return canonical, mol


# Test the pipeline on messy inputs
messy_inputs = [
    ("Sodium aspirin",     "CC(=O)OC1=CC=CC=C1C([O-])=O.[Na+]"),
    ("Metformin HCl",      "CN(C)C(=N)NC(=N)N.[Cl-]"),
    ("Invalid SMILES",     "not_a_molecule"),
    ("Charged ibuprofen",  "CC(C)CC1=CC=C(C=C1)C(C)C([O-])=O"),
    ("Clean caffeine",     "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"),
]

print("Curation pipeline results:\n")
for name, smi in messy_inputs:
    canonical, mol = curate_smiles(smi)
    if canonical:
        mw = f"MW={Descriptors.MolWt(mol):.1f}"
        print(f"  {name:25s}  ->  {canonical:40s}  {mw}")
    else:
        print(f"  {name:25s}  ->  REJECTED")

## Summary

This notebook covered the essential plumbing for working with molecular datasets in RDKit:

| Topic | Key tools | Wet-lab analogy |
|---|---|---|
| **SMILES / CSV reading** | `MolFromSmiles`, pandas | Reading compound IDs from a spreadsheet |
| **SDF reading / writing** | `SDMolSupplier`, `SDWriter`, `ForwardSDMolSupplier` | Exchanging compound data with a CRO |
| **PandasTools** | `AddMoleculeColumnToFrame` | Compound database with structure images |
| **Sanitization** | `MolFromSmiles` returns `None` on failure | Checking reagent purity before use |
| **Salt stripping** | `SaltRemover` | Isolating the free base / free acid |
| **Charge neutralization** | `rdMolStandardize.Uncharger` | Working at a reference pH |
| **Deduplication** | Canonical SMILES + `drop_duplicates` | Inventorying your freezer for duplicates |

### Key takeaways

1. **Always check for `None`** after parsing — silent failures are the norm, not the exception.
2. **Curate before computing** — salt forms, charges, and duplicates will skew every downstream analysis.
3. **Use `ForwardSDMolSupplier`** for large files (millions of compounds) to avoid memory issues.
4. **Canonical SMILES** is your deduplication key — different SMILES strings can encode the same molecule.

### Next up

**Notebook 09** will build on these data-handling skills to explore chemical space analysis — clustering, diversity selection, and library comparison.